<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/TOPO_CBP_LLAMA2GGUF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://huggingface.co/frankmorales2020/llama-3.1-70b-topo-governed-cbp-rh

## setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/

total 36K
drwx------ 40 root root 4.0K Nov  2 13:12 ..
drwx------  2 root root 4.0K Nov  2 11:04 llama.cpp
drwx------  2 root root 4.0K Nov  1 17:34 tmp
drwx------  2 root root 4.0K Nov  1 12:22 base_model_70b
drwx------  2 root root 4.0K Nov  1 12:22 checkpoints
drwx------  2 root root 4.0K Oct 26 15:10 gguf
drwx------  2 root root 4.0K Oct 19 19:52 hf_cache
drwx------  2 root root 4.0K Oct 15 01:11 mixtral-8x22b-topocbp-nolora
drwx------  9 root root 4.0K Oct  9 10:40 .


In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   99M  236G   1% /


## convert_hf_to_gguf

In [ ]:
%cd /content/drive/MyDrive/topogpt/llama.cpp

!PYTHONPATH=. python3 -u convert_hf_to_gguf.py \
    /content/drive/MyDrive/topogpt/base_model_70b \
    --outfile /content/temp_70b_bf16.gguf \
    --outtype bf16

## llama-quantize

In [ ]:
!chmod +x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize

In [ ]:
import os
import subprocess

TEMP_BF16 = "/content/temp_70b_bf16.gguf"
DRIVE_OUT = "/content/drive/MyDrive/topogpt"
LLAMA_DIR = os.path.join(DRIVE_OUT, "llama.cpp")
BUILD_DIR = os.path.join(LLAMA_DIR, "build")
QUANT_BIN = os.path.join(BUILD_DIR, "bin/llama-quantize")
FINAL_Q4  = os.path.join(DRIVE_OUT, "llama-3.1-70b-topo-governed-q4_k_m.gguf")

# Inject build lib paths so the dynamic linker finds all compiled .so files
env = os.environ.copy()
lib_paths = [
    os.path.join(BUILD_DIR, "bin"),
    os.path.join(BUILD_DIR, "src"),
    os.path.join(BUILD_DIR, "ggml/src"),
    BUILD_DIR,
    env.get("LD_LIBRARY_PATH", "")
]
env["LD_LIBRARY_PATH"] = ":".join(p for p in lib_paths if p)

print("=" * 80)
print("Quantizing /content/temp_70b_bf16.gguf to Q4_K_M...")
print("=" * 80)

proc = subprocess.Popen(
    [QUANT_BIN, TEMP_BF16, FINAL_Q4, "Q4_K_M"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

for line in proc.stdout:
    print(line, end="")

proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"llama-quantize failed with exit code {proc.returncode}")

# Verify final model artifact before removing temp file
if os.path.exists(FINAL_Q4) and os.path.getsize(FINAL_Q4) > 35 * (1024**3):
    final_gb = os.path.getsize(FINAL_Q4) / (1024**3)
    print("\n" + "=" * 80)
    print(f"SUCCESS: Generated {FINAL_Q4} ({final_gb:.2f} GB)")
    print("Purging local 141 GB temporary BF16...")
    os.remove(TEMP_BF16)
    print("✓ Local disk cleared.")
    print("=" * 80)
else:
    raise RuntimeError(f"Output artifact {FINAL_Q4} is missing or undersized.")

Quantizing /content/temp_70b_bf16.gguf to Q4_K_M...
version: 0.8.0-dev (build 1, commit 5969566)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '/content/temp_70b_bf16.gguf' to '/content/drive/MyDrive/topogpt/llama-3.1-70b-topo-governed-q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 32 key-value pairs and 724 tensors from /content/temp_70b_bf16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_p f32              = 0.900000
llama_model_loader: - kv   3:                      general.sampling.temp f32              = 0.600000
llama_model_loader: - kv   4:                               general.name str            

## inference-local

In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
  --no-cache-dir

In [ ]:
import sys
from llama_cpp import Llama

MODEL_PATH = "/content/drive/MyDrive/topogpt/llama-3.1-70b-topo-governed-q4_k_m.gguf"

print("Loading model...")
print(f"Model path: {MODEL_PATH}")




# 1. Initialize with stable CPU execution and verbose logs
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048,
    n_threads=4,       # Match allocated CPU threads
    n_gpu_layers=0,    # 0 avoids GPU VRAM overflow/thrashing
    use_mmap=False,
    verbose=False,   # Suppress verbose logs
)

# 2. Formatted prompt without the duplicate leading BOS tag
prompt = "<|start_header_id|>system<|end_header_id|>\n\nYou are an expert in safety-critical, deterministic systems engineering.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nExplain the concept of deterministic AI in safety-critical systems.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

print("\n--- Model Output (Streaming) ---")

# 3. Stream tokens in real time directly to stdout
stream = llm(
    prompt,
    max_tokens=1024,
    temperature=0.2,
    top_p=0.9,
    stop=["<|eot_id|>", "<|end_of_text|>"],
    stream=True
)

for chunk in stream:
    token_text = chunk["choices"][0]["text"]
    sys.stdout.write(token_text)
    sys.stdout.flush()

print()

Loading model...
Model path: /content/drive/MyDrive/topogpt/llama-3.1-70b-topo-governed-q4_k_m.gguf

--- Model Output (Streaming) ---
In safety-critical systems, deterministic AI refers to the use of artificial intelligence (AI) and machine learning (ML) techniques that provide predictable, reliable, and explainable results, ensuring the system's behavior is consistent and trustworthy. Deterministic AI is essential in safety-critical systems, such as those found in aerospace, automotive, healthcare, and industrial control systems, where human lives or significant economic losses are at stake.

**Key characteristics of deterministic AI:**

1. **Predictability**: The system's behavior is predictable and can be formally verified, ensuring that the AI system will always respond in a known and expected way.
2. **Reliability**: The system is designed to operate reliably, with minimal errors or failures, and can recover from faults or anomalies.
3. **Explainability**: The system's decision-ma

## hf

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, create_repo

# Retrieve your token from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')
user_or_name = 'frankmorales2020'

REPO_NAME = "llama-3.1-70b-topo-governed-gguf"
REPO_ID = f"{user_or_name}/{REPO_NAME}"
MODEL_PATH = "/content/drive/MyDrive/topogpt/llama-3.1-70b-topo-governed-q4_k_m.gguf"
FILENAME = os.path.basename(MODEL_PATH)

print(f"Target repository: {REPO_ID}")
print(f"Source file: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / (1024**3):.2f} GB)")

api = HfApi(token=HF_TOKEN)

# Create repository if it doesn't already exist
create_repo(
    repo_id=REPO_ID,
    token=HF_TOKEN,
    repo_type="model",
    exist_ok=True,
    private=False
)
print("✓ Repository confirmed.")

# Upload with native multi-part chunking and progress bar
print(f"Uploading {FILENAME} to {REPO_ID}...")
api.upload_file(
    path_or_fileobj=MODEL_PATH,
    path_in_repo=FILENAME,
    repo_id=REPO_ID,
    repo_type="model"
)

print(f"\n✓ Successfully published: https://huggingface.co/{REPO_ID}")

Target repository: frankmorales2020/llama-3.1-70b-topo-governed-gguf
Source file: /content/drive/MyDrive/topogpt/llama-3.1-70b-topo-governed-q4_k_m.gguf (39.60 GB)
✓ Repository confirmed.
Uploading llama-3.1-70b-topo-governed-q4_k_m.gguf to frankmorales2020/llama-3.1-70b-topo-governed-gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...topo-governed-q4_k_m.gguf:   0%|          |  120MB / 42.5GB            

Validating                    :           |   0%            


✓ Successfully published: https://huggingface.co/frankmorales2020/llama-3.1-70b-topo-governed-gguf


## inference-hf

In [ ]:
import sys
from google.colab import userdata
from llama_cpp import Llama
from warnings import filterwarnings

filterwarnings("ignore")

# Retrieve your token from Colab Secrets

HF_TOKEN = userdata.get('HF_TOKEN')

# Download from Hugging Face and initialize in one call
llm = Llama.from_pretrained(
    repo_id="frankmorales2020/llama-3.1-70b-topo-governed-gguf",
    filename="llama-3.1-70b-topo-governed-q4_k_m.gguf",
    hf_token=HF_TOKEN,
    n_ctx=2048,
    n_threads=4,       # Match your allocated vCPUs
    n_gpu_layers=0,    # 0 for CPU; adjust if sufficient VRAM (>= 40 GB) is available
    use_mmap=False,
    verbose=False
)

messages = [
    {
        "role": "system",
        "content": "You are an expert in safety-critical, deterministic systems engineering."
    },
    {
        "role": "user",
        "content": "Explain the concept of deterministic AI in safety-critical systems."
    }
]

response = llm.create_chat_completion(
    messages=messages,
    max_tokens=1024,
    temperature=0.2,
    top_p=0.9,
    stream=True
)

for chunk in response:
    delta = chunk["choices"][0].get("delta", {})
    if "content" in delta:
        sys.stdout.write(delta["content"])
        sys.stdout.flush()
print()

./llama-3.1-70b-topo-governed-q4_k_m.ggu(…): reconstructing file:   0%|          |  0.00B / 42.5GB            

./llama-3.1-70b-topo-governed-q4_k_m.ggu(…): downloading bytes:           |  0.00B            

Deterministic AI in safety-critical systems refers to the use of artificial intelligence (AI) and machine learning (ML) techniques that provide predictable, reliable, and reproducible results in systems where safety is paramount. The primary goal of deterministic AI is to ensure that the system's behavior is consistent, transparent, and explainable, which is critical in safety-critical domains such as aerospace, automotive, healthcare, and industrial control systems.

Key characteristics of deterministic AI in safety-critical systems:

1. **Predictability**: The system's behavior is predictable and can be formally verified, ensuring that it will always respond in a known and expected way to a given input.
2. **Reproducibility**: The system's output is reproducible, meaning that it will always produce the same result given the same input and conditions.
3. **Explainability**: The system's decision-making process is transparent and explainable, allowing developers and operators to unders